# Public slideshow refresh — v2-aware

Refreshes `datateam_portfolio_public` (the public, read-only Feature Service that powers the lobby-display slideshow) from the authoritative v2 sources. Schema-self-healing: when `PROJECT_FIELDS` / `TASK_FIELDS` are widened, the missing columns are added to the existing tables in place (no delete-and-recreate, since `delete_from_definition` has been observed to fail on this service with `Object reference not set to an instance of an object`).

**Idempotent.** Safe to schedule as a recurring AGO Notebook task.

**To add a public field:** add the name to `PROJECT_FIELDS` or `TASK_FIELDS` below and run. The field must exist on the source v2 layer (the schema is copied verbatim from source to preserve type/length/domain).

**To remove a public field:** drop it from the list — but the column will stay in the public table until the service is recreated. (AGO's `delete_from_definition` is currently failing on this service.) Removal isn't critical; extra columns just sit there unused.

**Replaces:** `create_public_slideshow_copy.ipynb` (kept for reference).


In [ ]:
from arcgis.gis import GIS
from arcgis.features import FeatureLayer, FeatureLayerCollection

# 'home' resolves to the notebook owner when run inside AGO (manually or scheduled).
gis = GIS("home")
print(f"Authenticated as: {gis.users.me.username} @ {gis.url}")

## Config

Field lists are the union of what the slideshow needs today plus the new org-cleanup fields we want exposed publicly. Source URLs are the authoritative v2 services.

In [ ]:
SOURCE_PROJECTS_URL = "https://services3.arcgis.com/9coHY2fvuFjG9HQX/ArcGIS/rest/services/datateam_portfolio_v2/FeatureServer/0"
SOURCE_TASKS_URL    = "https://services3.arcgis.com/9coHY2fvuFjG9HQX/ArcGIS/rest/services/datateam_portfolio_v2/FeatureServer/1"

PUBLIC_SERVICE_NAME = "datateam_portfolio_public"
# Item ID of the existing public service (skip slow content search when set).
# Leave None on first-ever run; update after Phase 2 prints the id.
KNOWN_PUBLIC_ITEM_ID = "844b8de3880449e593b5d61e48c6629a"

PROJECT_FIELDS = [
    "project_number",       # primary key (JS aliases to .id)
    "title",                # deadline list, debugging
    "status",               # filtered on by nearly every slide
    "category",             # 'Projects by category' slide
    "end_date",             # fallback deadline (JS alias .end)
    "working_due",          # primary deadline
    "actual_end",           # completions: throughput, intake, overdue trend
    "owning_unit",          # org-cleanup unit field
    "owning_team",          # org-cleanup team field (also the DP team for DP projects)
    "is_data_program",      # explicit DP flag (replaces data_program_team for DP-status checks)
    "dp_goal",              # legacy DP-status fallback for any rows not yet flagged
]

TASK_FIELDS = [
    "task_number",          # primary key (JS alias .idx)
    "project_number",       # FK to projects (JS alias .project_id)
    "title",                # deadline list
    "status",               # filters everywhere
    "priority",             # 'Open task priority breakdown' slide
    "start_date",           # intake balance (JS alias .start)
    "due_date",             # fallback deadline (JS alias .due)
    "working_due",          # primary deadline
    "actual_end",           # completions: throughput, intake
]

# Editor-tracking and admin metadata never copied to public, even if added
# to a *_FIELDS list above.
ALWAYS_DROP = {"CreationDate", "Creator", "EditDate", "Editor", "GlobalID"}

## Phase 1 — Open source layers

Confirms auth + prints row counts so you can sanity-check the public copy at the end.

In [ ]:
src_projects = FeatureLayer(SOURCE_PROJECTS_URL, gis=gis)
src_tasks    = FeatureLayer(SOURCE_TASKS_URL,    gis=gis)

# Show both totals so you can see how many rows are soft-deleted (and therefore
# excluded from the public copy by Phase 4's `where=deleted_at IS NULL` filter).
src_projects_total   = src_projects.query(where="1=1", return_count_only=True)
src_projects_kept    = src_projects.query(where="deleted_at IS NULL", return_count_only=True)
src_tasks_total      = src_tasks.query(where="1=1", return_count_only=True)
src_tasks_kept       = src_tasks.query(where="deleted_at IS NULL", return_count_only=True)
print(f"Source projects: {src_projects_kept} kept / {src_projects_total} total ({src_projects_total - src_projects_kept} soft-deleted)")
print(f"Source tasks   : {src_tasks_kept} kept / {src_tasks_total} total ({src_tasks_total - src_tasks_kept} soft-deleted)")

## Phase 2 — Find or create the public service

Looks up by known item ID first (fast), then falls back to title+owner search, then to broader name search, then creates as a last resort.

In [ ]:
def get_or_create_public_service(name: str):
    # 0. Direct lookup by known item ID (most reliable).
    if KNOWN_PUBLIC_ITEM_ID:
        item = gis.content.get(KNOWN_PUBLIC_ITEM_ID)
        if item and item.type == "Feature Service":
            print(f"Public service found by ID: {item.id} ({item.title}, owner={item.owner})")
            return item

    # 1. Exact title match owned by current user.
    me = gis.users.me.username
    hits = [
        i for i in gis.content.search(f'title:"{name}" owner:{me}', item_type="Feature Service")
        if i.title == name
    ]
    if hits:
        print(f"Public service exists (owned by me): {hits[0].id} ({hits[0].title})")
        return hits[0]

    # 2. Fallback: search the org by name and match via URL substring. Handles
    # the case where the service was created by another user, or has a display
    # title that differs from its service name.
    target_url_substr = f"/{name}/FeatureServer"
    for i in gis.content.search(name, item_type="Feature Service", max_items=50):
        if target_url_substr in (i.url or ""):
            print(f"Public service exists (found by URL): {i.id} ({i.title}, owner={i.owner})")
            return i

    # 3. Truly missing — create it.
    print(f"Creating new public service: {name}")
    item = gis.content.create_service(
        name=name,
        service_description="Read-only public copy of selected portfolio fields for the analytics-tracker lobby-display slideshow. Refreshed by scheduled notebook.",
        has_static_data=False,
        max_record_count=4000,
        capabilities="Query",
        service_type="featureService",
    )
    print(f"Created: {item.id}")
    return item

public_item = get_or_create_public_service(PUBLIC_SERVICE_NAME)
public_flc  = FeatureLayerCollection.fromitem(public_item)

## Phase 3 — Schema sync (add missing fields, don't delete)

For each source table, build a target definition with the kept fields. If the target table doesn't exist, create it. If it exists but is missing one or more fields, add them in place — copying the field definition verbatim from source so type/length/domain/alias are preserved.

**Why add-in-place and not delete-and-recreate:** `delete_from_definition` has been observed to fail on this service with `Object reference not set to an instance of an object` (AGO-side bug, not our payload). Add-in-place is forward-only — extra columns left behind after a field removal are accepted as a tradeoff.

In [ ]:
def build_table_def(src_layer, keep_fields, new_name):
    src_props = dict(src_layer.properties)
    src_fields = src_props.get("fields", [])
    kept = []
    for f in src_fields:
        nm = f["name"]
        if nm in ALWAYS_DROP:
            continue
        if nm == "ObjectId" or nm in keep_fields:
            kept.append(dict(f))
    return {
        "name": new_name,
        "type": "Table",
        "displayField": src_props.get("displayField") or kept[0]["name"],
        "description": f"Public slideshow copy of {src_props.get('name')} \u2014 slideshow-only fields.",
        "objectIdField": "ObjectId",
        "fields": kept,
        "capabilities": "Query",
        "supportsAdvancedQueries": True,
        "hasAttachments": False,
    }

def ensure_table(item_id: str, src_layer, keep_fields, new_name):
    flc = FeatureLayerCollection.fromitem(gis.content.get(item_id))
    existing = next((t for t in flc.tables if t.properties.name == new_name), None)

    if existing:
        existing_field_names = {f["name"] for f in existing.properties.fields}
        expected_field_names = set(keep_fields) | {"ObjectId"}
        missing = expected_field_names - existing_field_names

        if missing:
            # Add missing fields in place (no delete).
            src_field_defs = {f["name"]: dict(f) for f in src_layer.properties.fields}
            to_add = [src_field_defs[n] for n in sorted(missing) if n in src_field_defs]
            unavailable = sorted(missing - {f["name"] for f in to_add})
            if to_add:
                print(f"  Adding fields to '{new_name}': {[f['name'] for f in to_add]}")
                existing.manager.add_to_definition({"fields": to_add})
            if unavailable:
                print(f"  WARNING: cannot add {unavailable} (not present on source layer)")
            return existing
        else:
            print(f"  Table '{new_name}' schema is up to date ({len(existing.properties.fields)} fields)")
            return existing

    # Table doesn't exist — create from source.
    tdef = build_table_def(src_layer, keep_fields, new_name)
    print(f"  Creating table '{new_name}' ({len(tdef['fields'])} fields)")
    flc.manager.add_to_definition({"tables": [tdef]})
    flc = FeatureLayerCollection.fromitem(gis.content.get(item_id))
    return next(t for t in flc.tables if t.properties.name == new_name)

print("Syncing schema...")
tgt_projects = ensure_table(public_item.id, src_projects, PROJECT_FIELDS, "projects")
tgt_tasks    = ensure_table(public_item.id, src_tasks,    TASK_FIELDS,    "tasks")

## Phase 4 — Refresh data (truncate + append)

Truncate-then-append keeps the public service's item ID and URLs stable across refreshes. Appends in 1,000-row chunks to stay under AGO's per-request limits.

In [ ]:
def refresh(src_layer, tgt_layer, fields, label):
    keep_set = set(fields) | {"ObjectId"}
    out_fields = ",".join(sorted(keep_set))
    # Filter out soft-deleted source rows so the public copy stays in sync
    # with what the internal app shows. Internal queries gate on
    # `deleted_at IS NULL` (see src/data-bootstrap.js); the public copy must
    # do the same, otherwise items deleted in-app keep showing on the
    # lobby slideshow until the table is recreated. Both projects and
    # tasks layers have the soft-delete columns.
    fset = src_layer.query(where="deleted_at IS NULL", out_fields=out_fields, return_geometry=False)
    raw = fset.features
    cleaned = [
        {"attributes": {k: v for k, v in f.attributes.items() if k in keep_set and k != "ObjectId"}}
        for f in raw
    ]
    tgt_layer.manager.truncate()
    added = 0
    for i in range(0, len(cleaned), 1000):
        chunk = cleaned[i:i + 1000]
        result = tgt_layer.edit_features(adds=chunk)
        ok = sum(1 for r in result.get("addResults", []) if r.get("success"))
        added += ok
        if ok != len(chunk):
            fails = [r for r in result.get("addResults", []) if not r.get("success")]
            print(f"  ! {label}: only {ok}/{len(chunk)} added in chunk starting at {i}")
            for fr in fails[:3]:
                print(f"    fail sample: {fr}")
    print(f"  {label}: {added}/{len(cleaned)} rows loaded (soft-deleted excluded)")
    return added

print("Refreshing data...")
n_proj = refresh(src_projects, tgt_projects, PROJECT_FIELDS, "projects")
n_task = refresh(src_tasks,    tgt_tasks,    TASK_FIELDS,    "tasks")

## Phase 5 — Share publicly

Idempotent: only flips sharing if it isn't already public.

In [ ]:
public_item = gis.content.get(public_item.id)  # refresh metadata
if public_item.access != "public":
    print("Sharing with Everyone...")
    public_item.share(everyone=True)
    print("Shared.")
else:
    print("Already shared with Everyone.")

## Phase 6 — Verification

Prints final row counts, item id, and the per-table URLs to confirm `ARCGIS_CONFIG.publicProjectsUrl` / `publicTasksUrl` in `src/agol.js` still match.

In [ ]:
flc = FeatureLayerCollection.fromitem(gis.content.get(public_item.id))
print("Public service:")
print(f"  item id: {public_item.id}")
print(f"  access : {public_item.access}")
print()
print("Per-table URLs (paste into ARCGIS_CONFIG in src/agol.js):")
for t in flc.tables:
    count = t.query(where="1=1", return_count_only=True)
    field_names = [f.name for f in t.properties.fields]
    print(f"  {t.properties.name:10s}  rows={count:4d}  fields={len(field_names)}")
    print(f"             url={t.url}")